# Bring your own CNN: a trained U-Net from checkpoint to Earth Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)

This notebook takes a U-Net you trained with
[segmentation-models-pytorch](https://github.com/qubvel-org/segmentation_models.pytorch)
(SMP) and runs it inside Earth Engine on your own imagery. The steps:

1. load your checkpoint and check that TensorEE supports the model;
2. work out how much extra border (the halo) each tile needs, and plan the tiles;
3. upload the model's weights to Earth Engine once, as table assets;
4. predict one tile and compare it with PyTorch on the same pixels;
5. export a whole region.

**Offline first.** With the settings as they are, the notebook uses a small demonstration
U-Net with random weights and runs every step offline: it builds the Earth Engine
computations but sends nothing. Then you set your checkpoint, your project and your input,
and turn on the billed steps one at a time. Each billed step says beforehand what it creates
and what is known of its cost, and runs only when both `RUN_LIVE` and its own switch are
`True`.

**What has been checked.** Each building block this U-Net uses agrees with PyTorch at the
settings it was validated at. One trained U-Net with a ResNet-18 encoder, prepared the way
this notebook prepares yours, ran in Earth Engine on a 64 × 64 test input and matched PyTorch
within 1.2e-5 in the logits. The tiled recipe in this notebook has **not** been run in Earth
Engine as a whole: step 9's comparison is how you check it for your model. Agreement with
PyTorch says nothing about accuracy, which you measure on your own labelled data.

## 1. Install

The `segwater` extra brings segmentation-models-pytorch 0.5.0, the version the library was
checked with. PyTorch is already installed in Colab.

In [ ]:
%pip install --quiet "tensor-ee[segwater]"

## 2. Settings

Edit this cell, and only this cell.

**Your model.** `CHECKPOINT` is the path of your trained weights, saved with
`torch.save(model.state_dict(), path)`. In Colab, upload it with the **Files** panel on the left.
Leave it `None` to use the small demonstration model, which runs offline only. `ENCODER`,
`BANDS` and `CLASSES` must match how you trained the model.

**Your input.** `INPUT_IMAGE` is an Earth Engine image with the bands `BANDS`, already scaled
the way your training data was (TensorEE does not normalize for you). `CRS`, `REGION_X`,
`REGION_Y` and `PIXEL` place the region: the upper-left corner of its first tile, in metres of
`CRS`. The image must also cover a border of one halo around the region (step 5 prints how
wide).

**The billed steps.** Each runs only when `RUN_LIVE` and its own switch are both `True`.

After changing `RUN_LIVE`, restart the runtime (in Colab: **Runtime → Restart session**) and
run the notebook again from the top: the offline mode stays installed until the runtime
restarts.

In [ ]:
PROJECT = "YOUR_EE_PROJECT"   # your Cloud project registered for Earth Engine
RUN_LIVE = False              # True: sign in; the switches below then choose the billed steps

# Your model
CHECKPOINT = None             # for example "my_unet.pth"; None = demonstration model, offline only
ENCODER = "resnet34"
BANDS = ("VV", "VH")          # your input bands, in training order
CLASSES = 2

# Your input and the region
INPUT_IMAGE = "projects/YOUR_EE_PROJECT/assets/YOUR_PREPARED_INPUT"
CRS = "EPSG:32651"
REGION_X, REGION_Y = 286030, 2640860   # upper-left corner of the region, in CRS metres
PIXEL = 10                             # metres per pixel
TILE = 1024                            # tile side for the region, in pixels
REGION_COLUMNS, REGION_ROWS = 2, 1     # the region is this many tiles
TEST_TILE = 256                        # side of the single test tile of step 8

# What the live steps create
WEIGHTS_MANIFEST = "my_unet_weights.json"   # a local file recording the uploaded weights
TEST_ASSET = "my_unet_test_tile"            # new image assets in your project
REGION_ASSET = "my_unet_region"

# The billed steps
UPLOAD_WEIGHTS = False     # step 7
PREDICT_TEST_TILE = False  # step 8
COMPARE_TEST_TILE = False  # step 9
EXPORT_REGION = False      # step 10

In [ ]:
PLACEHOLDER = "YOUR_EE_PROJECT"


def live_step(what, switch=True):
    """True when a billed step should run; refuses a placeholder project or a missing checkpoint."""
    if not (RUN_LIVE and switch):
        reason = "RUN_LIVE is False" if not RUN_LIVE else "its switch is False"
        print(f"Skipped: {what}. {reason}, so nothing was sent to Earth Engine.")
        return False
    if not PROJECT or PROJECT == PLACEHOLDER:
        raise ValueError("Set PROJECT to your own Cloud project ID in the settings cell.")
    if CHECKPOINT is None:
        raise ValueError("Live steps need your trained checkpoint: set CHECKPOINT. "
                         "The demonstration model has random weights.")
    return True

## 3. Sign in, or work offline

With `RUN_LIVE = True` the first cell signs you in; the first time, `ee.Authenticate()` shows a
link or a prompt. Signing in costs nothing by itself. With `RUN_LIVE = False` the second cell
installs the Earth Engine client's offline function signatures (`ee.apitestcase.ApiTestCase`,
a test helper that ships with the client), so the notebook can build Earth Engine computations
without credentials and without sending anything.

In [ ]:
import ee

if live_step("connect to Earth Engine"):
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print("Signed in. Requests are billed to", PROJECT)

In [ ]:
import ee

if not RUN_LIVE:
    from ee.apitestcase import ApiTestCase

    offline = ApiTestCase()
    offline.setUp()   # offline signatures: no credentials, no network
    print("Offline: the notebook builds Earth Engine computations but sends none.")

## 4. Load the model

With `CHECKPOINT = None` this builds a shallow U-Net (three encoder stages, small decoder)
with random weights, so that the offline steps stay quick. With your checkpoint, it builds the
standard U-Net and loads your weights strictly: a missing or unexpected weight name stops it,
which usually means the settings do not match your training.

In [ ]:
import torch
import segmentation_models_pytorch as smp

torch.set_num_threads(2)


def make_model(**changes):
    settings = dict(encoder_name=ENCODER, encoder_weights=None, in_channels=len(BANDS),
                    classes=CLASSES, activation=None)
    settings.update(changes)
    return smp.Unet(**settings).eval()


if CHECKPOINT is None:
    torch.manual_seed(20261020)
    model = make_model(encoder_depth=3, decoder_channels=(16, 8, 4))
    tile, test_tile = 64, 64      # small tiles for the demonstration
    print("Demonstration model: a shallow U-Net with random weights.")
else:
    model = make_model()
    model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu", weights_only=True), strict=True)
    tile, test_tile = TILE, TEST_TILE
    print("Loaded", CHECKPOINT)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 5. Check support, and find the halo

`adapt_smp_model` prepares the model for TensorEE at a fixed input size, and checks that the
prepared model gives exactly the same numbers as yours on a test input. `check_compatibility`
then checks every operation. (Notebook 02 explains these reports in more detail.)

`analyze_receptive_field` works out how far each output pixel can see into the input. That
reach is the **halo**: to compute a tile exactly as the model would compute it on a larger
image, Earth Engine must read the tile plus a halo on every side, and the halo is thrown away
afterwards. For a standard U-Net with a ResNet-34 encoder the halo is 544 pixels, so a
1,024-pixel tile reads 2,112 × 2,112 pixels: about 4.25 input pixels per output pixel. Larger
tiles waste less.

In [ ]:
from tensor_ee import check_compatibility
from tensor_ee.receptive_field import analyze_receptive_field
from tensor_ee.smp import adapt_smp_model


def prepare(height, width):
    """The model adapted to one input size, refused unless every operation is supported."""
    shape = (1, len(BANDS), height, width)
    adapted = adapt_smp_model(model, input_shape=shape)
    report = check_compatibility(adapted, shape)
    if report.status != "supported":
        raise ValueError(f"TensorEE does not support this model: {report.to_dict()['blockers'][:3]}")
    return adapted


probe = prepare(64, 64)
support = analyze_receptive_field(probe)
halo, alignment = support.safe_halo, support.alignment
print("Supported. Prepared model equals the original on a test input:",
      probe.tensor_ee_smp_metadata["torch_equal"])
print(f"Halo: {halo} pixels on each side ({halo * PIXEL / 1000:g} km). "
      f"Tile sizes and positions must be multiples of {alignment} pixels.")

## 6. Plan the tiles

The plan puts the region's tiles side by side and gives each one its halo. Every tile then
reads an input of the same size, which the model needs. The region's input must cover the
region plus the halo on every side.

There are two plans: one test tile, for step 8, and the region, for step 10.

In [ ]:
from tensor_ee import Grid
from tensor_ee.converter import PixelWindow, plan_tiles


def plan_region(side, columns, rows):
    """Tiles of side x side pixels, with a full halo of real input around the whole region."""
    for name, value in (("tile side", side), ("halo", halo)):
        if value % alignment:
            raise ValueError(f"The {name} ({value}) must be a multiple of {alignment} pixels.")
    width, height = columns * side + 2 * halo, rows * side + 2 * halo
    padded = Grid(CRS, (PIXEL, 0, REGION_X - halo * PIXEL, 0, -PIXEL, REGION_Y + halo * PIXEL),
                  width, height)
    region = PixelWindow(halo, halo, columns * side, rows * side)
    return plan_tiles(padded, side, halo, alignment=alignment, roi=region)


def describe(plan, name):
    context = plan.tiles[0].context
    read = sum(t.context.pixels for t in plan.tiles)
    print(f"{name}: {len(plan.tiles)} tile(s) of {plan.core_size[0]} x {plan.core_size[1]} pixels; "
          f"each reads {context.width} x {context.height}; "
          f"{read / plan.roi.pixels:.2f} input pixels per output pixel.")
    print(f"  Input needed: {plan.global_grid.width} x {plan.global_grid.height} pixels "
          f"from corner {plan.global_grid.transform[2]}, {plan.global_grid.transform[5]} in {CRS}.")


test_plan = plan_region(test_tile, 1, 1)
region_plan = plan_region(tile, REGION_COLUMNS, REGION_ROWS)
describe(test_plan, "Test tile")
describe(region_plan, "Region")

## 7. Upload the weights once (billed to your project)

TensorEE folds each BatchNorm layer into the convolution before it (the same arithmetic, fewer
steps) and stores the resulting numbers in Earth Engine table assets. Every prediction then
reads them from there, so they are uploaded once and reused.

**What this creates.** Table assets in your project, under a new name
`projects/PROJECT/assets/tensor_ee_weights_<random>`, each holding at most 8 MB of numbers as
text; the cell prints an estimate of how many. One batch table-export task per table, at most
two at a time; the cell waits for all of them. Locally, the file `WEIGHTS_MANIFEST`, which
records the tables and a fingerprint of every convolution. Keep that file: the later steps
need it, and in Colab it disappears when the runtime ends, so download it from the **Files**
panel.

**What it costs.** Not recorded. The trained ResNet-18 U-Net of notebook 01 has 14.3 million
numbers and took 29 tables.

In [ ]:
import math

from tensor_ee import effective_convolutions

parameters = effective_convolutions(probe)
values = sum(p.weights.size + (0 if p.bias is None else p.bias.size) for p in parameters)
print(f"{len(parameters)} folded convolutions, {values:,} numbers: "
      f"about {math.ceil(values * 14 / 7_000_000)} tables.")

In [ ]:
import json
import uuid
from collections import Counter
from pathlib import Path

from tensor_ee import export_weights

if live_step("upload the weights", UPLOAD_WEIGHTS):
    manifest_path = Path(WEIGHTS_MANIFEST)
    if manifest_path.exists():
        raise FileExistsError(f"{manifest_path} already exists: the weights are uploaded. "
                              "Use a new WEIGHTS_MANIFEST name only for new weights.")
    namespace = f"projects/{PROJECT}/assets/tensor_ee_weights_{uuid.uuid4().hex[:12]}"

    def progress(manifest):
        states = Counter(shard["status"] for shard in manifest["shards"])
        print(json.dumps({"status": manifest["status"], "tables": dict(states)}), flush=True)

    export_weights(parameters, namespace, manifest_path, on_update=progress)
    print("Weights uploaded; manifest written to", manifest_path)

## 8. Predict the test tile (billed to your project)

The next cell builds the prediction: the model, compiled to Earth Engine operations for the test tile's input size, with
its weights read from the uploaded tables. Offline, the tables are placeholders that do not
exist, which is enough to build and measure the computation.

The result has these bands: `score_0`, `score_1`, … (the raw logits), `p_0`, `p_1`, …
(probabilities), `class_index` (the most likely class) and, for two classes, `binary_class`
(1 where `p_1` ≥ 0.5). A prediction is made only where every input band is present.

In [ ]:
import json
from pathlib import Path

from tensor_ee import MatchedWeightProvider, Raster, binary_threshold, segmentation_outputs
from tensor_ee.converter import compile_tiled_model
from tensor_ee.converter.request import deep_recursion

SCORES = tuple(f"score_{k}" for k in range(CLASSES))
PROBABILITIES = tuple(f"p_{k}" for k in range(CLASSES))


def weights_manifest():
    """The uploaded weights, or offline a stand-in that names tables which do not exist."""
    if RUN_LIVE and Path(WEIGHTS_MANIFEST).exists():
        return json.loads(Path(WEIGHTS_MANIFEST).read_text())
    if RUN_LIVE:
        raise FileNotFoundError(f"{WEIGHTS_MANIFEST} not found: upload the weights first (step 7).")
    return {"format": "native_ee_featurecollection_weights_v2", "status": "ready",
            "layers": {p.key: {"shape": list(p.shape), "dtype": p.dtype, "bias": p.bias is not None,
                               "source": p.source, "sha256": p.sha256,
                               "assets": ["projects/YOUR_EE_PROJECT/assets/weights_not_uploaded"]}
                       for p in parameters}}


def input_image():
    if RUN_LIVE:
        return ee.Image(INPUT_IMAGE)
    return ee.Image.constant([0] * len(BANDS)).rename(list(BANDS))   # offline stand-in


def build_prediction(plan):
    """The model's Earth Engine computation over a plan's tiles, with its output bands."""
    context = plan.tiles[0].context
    adapted = prepare(context.height, context.width)
    if analyze_receptive_field(adapted).safe_halo != halo:
        raise ValueError("The halo changed with the input size; revise the plan.")
    provider = MatchedWeightProvider(adapted, weights_manifest())
    source = input_image().select(list(BANDS)).toFloat().reproject(plan.global_grid.projection())
    valid = Raster(source.mask().reduce(ee.Reducer.min()).rename("valid"), ("valid",), plan.global_grid)
    logits = compile_tiled_model(adapted, source, BANDS, plan, weight_provider=provider)
    outputs = segmentation_outputs(logits, logit_bands=SCORES, probability_bands=PROBABILITIES,
                                   validity=valid)
    bands = [outputs.logits.image, outputs.probabilities.image, outputs.classes.image]
    if CLASSES == 2:
        bands.append(binary_threshold(outputs.probabilities, positive_class=1, threshold=0.5,
                                      output_band="binary_class").image)
    return ee.Image.cat(bands).toFloat()   # one type for export; class values stay exact


def export_task(image, plan, asset_name, description):
    grid = plan.roi_grid
    return ee.batch.Export.image.toAsset(
        image=image, description=description, assetId=f"projects/{PROJECT}/assets/{asset_name}",
        region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
        maxPixels=grid.width * grid.height, pyramidingPolicy={".default": "sample"}, overwrite=False)


with deep_recursion():
    test_prediction = build_prediction(test_plan)
    test_graph_bytes = len(test_prediction.serialize().encode())
print(f"Test tile computation: {test_graph_bytes:,} bytes of Earth Engine expression.")

**What the export creates.** One batch export task and, when it completes, one new image
asset `projects/PROJECT/assets/TEST_ASSET` over the test tile (not its halo).

**What it costs.** Not measured for this recipe. For scale: the trained water U-Net's
224 × 224 chip took 64.9 batch EECU-seconds, and this test tile reads a larger input
(step 6). If Earth Engine reports that it ran out of memory or time, reduce `TEST_TILE` and
try again; do not just resubmit.

In [ ]:
import time


def wait(task, minutes=60):
    """Check a task every 30 seconds until it ends."""
    deadline = time.time() + minutes * 60
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        if time.time() > deadline:
            raise TimeoutError("Still running; run this cell again later, or watch the Tasks tab.")
        time.sleep(30)
    print("Task", task.id, state)
    if state != "COMPLETED":
        raise RuntimeError(task.status().get("error_message", "the export did not complete"))


if live_step("export the test tile (1 batch task, 1 new asset)", PREDICT_TEST_TILE):
    test_task = export_task(test_prediction, test_plan, TEST_ASSET, "tensor_ee_test_tile")
    test_task.start()
    print("Started", test_task.id)
    wait(test_task)

## 9. Compare the test tile with PyTorch (billed to your project)

This is the check that matters for your model: Earth Engine's logits against your model's in
PyTorch, on exactly the same input pixels. The cell downloads the test tile's input (tile plus
halo) and the exported logits, runs your model on the input locally, and reports the largest
absolute difference. TensorEE's target is 1e-4.

**What this does.** Two interactive requests (`ee.data.computePixels`): the input and the
exported result. It creates nothing. The input must be complete over the tile and its halo.

**What it costs.** Small interactive requests, not measured. The cell refuses an input over
40 MB.

In [ ]:
import numpy as np


def download(image, grid, bands):
    """The exact float32 pixels of an image's bands on a grid: one interactive request."""
    names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
    values = ee.data.computePixels({
        "expression": image.select(list(bands)), "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": grid.width, "height": grid.height},
                 "affineTransform": dict(zip(names, grid.transform)), "crsCode": grid.crs}})
    return np.stack([values[band] for band in bands]).astype(np.float32)


if live_step("compare the test tile with PyTorch (2 requests)", COMPARE_TEST_TILE):
    padded = test_plan.global_grid
    size = padded.width * padded.height * (len(BANDS) + 1) * 4
    if size > 40_000_000:
        raise ValueError(f"The input is {size / 1e6:.0f} MB; reduce TEST_TILE.")
    source = ee.Image(INPUT_IMAGE).select(list(BANDS)).toFloat().reproject(padded.projection())
    valid = source.mask().reduce(ee.Reducer.min()).rename("valid")
    pixels = download(source.unmask(0).addBands(valid), padded, (*BANDS, "valid"))
    if pixels[-1].min() < 1:
        raise ValueError("The input has missing pixels in the tile or its halo; the comparison "
                         "needs complete input. Move the region or fill the gaps.")
    exported = download(ee.Image(f"projects/{PROJECT}/assets/{TEST_ASSET}"), test_plan.roi_grid, SCORES)

    with torch.no_grad():
        reference = model(torch.from_numpy(pixels[None, :-1].copy()))[0].numpy()
    core = reference[:, halo:halo + test_tile, halo:halo + test_tile]
    error = np.abs(exported - core)
    disagreements = int((exported.argmax(0) != core.argmax(0)).sum())
    print(f"Largest absolute logit difference: {error.max():.3e} (target 1e-4)")
    print(f"Mean absolute difference: {error.mean():.3e}; "
          f"class disagreements: {disagreements} of {core[0].size} pixels")
    print("Within the target." if error.max() <= 1e-4 else
          "Above the target: please report it, with these numbers, as an issue.")

## 10. Export the region (billed to your project)

**What this creates.** One batch export task and, when it completes, one new image asset
`projects/PROJECT/assets/REGION_ASSET` over the region (not its halo), with the bands of
step 8, in float32.

**What it costs.** Not measured. Earth Engine computes every tile with its halo (step 6
prints how many input pixels it reads per output pixel), and capacity at these tile sizes has
not been measured: an export can fail for lack of memory. Run the test tile first.

In [ ]:
if live_step("export the region (1 batch task, 1 new asset)", EXPORT_REGION):
    with deep_recursion():
        region_prediction = build_prediction(region_plan)
        region_task = export_task(region_prediction, region_plan, REGION_ASSET, "tensor_ee_region")
        region_task.start()
    print("Started", region_task.id, "- follow it in the Tasks tab, or wait here.")
    wait(region_task, minutes=240)

## What this shows, and what it does not

- **Offline**, the notebook shows that TensorEE supports the model and can build its Earth
  Engine computation for your tiles.
- **Step 9** is live agreement for your model, on one tile and one input. It is the evidence
  to keep, with its numbers.
- **The region** reuses the same computation for every tile; the halo is what makes tile
  borders match a single run of the model, as far as the model's reach goes.
- **Accuracy** is a separate question: evaluate the exported predictions against your own
  labelled data.

The [guide this notebook follows](https://github.com/noelEOS/tensor-ee/blob/main/docs/own-model.md)
gives the details: other encoders, irregular regions, and how validity masks are handled.
**Cleaning up**: the weight tables and image assets stay in your project until you delete them
in the Code Editor's **Assets** tab; keep the weight tables as long as you predict with them.